# Day K2 — 표본화의 출처와 모달 간 동기화: "500 Hz로 맞췄다"를 의심하기

설명은 학습 사이트(또는 원본 레슨)에서 읽고, 코드는 이 노트북에서 위에서 아래로 차례대로 실행하세요. 앞 셀이 만든 변수와 파일을 뒤 셀이 씁니다.

원본 레슨: [lessons/advanced/K-multimodal-fog/day-K2.md](https://github.com/evansuhohyun-927/neurodegeneration-bioinformatics/blob/main/lessons/advanced/K-multimodal-fog/day-K2.md)

In [ ]:
# 레슨 코드가 쓰는 폴더를 미리 만든다 (Colab에서는 /content 아래에 생긴다)
import os
for d in ["data", "figures", "reports", "results", "../data", "../figures", "../reports"]:
    os.makedirs(d, exist_ok=True)

## 예시로 이해하기

### 예시 1 — 업샘플은 정보를 늘리지 않는다: 100 Hz를 500 Hz로 보간한 스펙트럼

In [ ]:
import numpy as np
from scipy.interpolate import CubicSpline
from scipy.signal import welch

def make_acc(t, rng):
    """합성 데이터 뼈대의 가속도 식 (25–35초 FOG 가정)"""
    label = ((t >= 25) & (t < 35)).astype(int)
    return np.where(label == 0, np.sin(2*np.pi*1.0*t), 0.4*np.sin(2*np.pi*5.0*t)) + 0.1*rng.standard_normal(t.size)

# (가) 처음부터 500 Hz로 기록했다면
fs = 500
t = np.arange(0, 60, 1/fs)
acc500 = make_acc(t, np.random.default_rng(0))

# (나) 센서는 100 Hz로 기록했고, 나중에 cubic spline으로 500 Hz 격자에 맞췄다면
t100 = np.arange(0, 60, 1/100)
acc100 = make_acc(t100, np.random.default_rng(1))
up = CubicSpline(t100, acc100)(t[t <= t100[-1]])          # 500 Hz 격자로 보간

print("행 수: 500 Hz 원본 %d, 100 Hz 원본 %d, 보간 후 %d" % (acc500.size, acc100.size, up.size))
print("보간 결과의 5행마다 값 == 100 Hz 원본:", np.allclose(up[::5], acc100))
for name, x in [("처음부터 500 Hz", acc500), ("100 Hz → 500 Hz 보간", up)]:
    f, p = welch(x, fs=fs, nperseg=1000)                  # 2초 window, 0.5 Hz 간격
    share = p[f > 50].sum() / p.sum()
    print(f"{name:16s} 50 Hz 위 power 비율 = {share:.2e}")

### 예시 2 — 한 행씩 건너뛰면 300 Hz가 200 Hz로 접힌다

In [ ]:
import numpy as np
from scipy.signal import welch, decimate

# 1000 Hz로 기록한 "EMG 같은" 신호: 80 Hz 성분 + 300 Hz 성분 (설명용 가정)
fs_hi = 1000
t = np.arange(0, 10, 1/fs_hi)
rng = np.random.default_rng(0)
x = np.sin(2*np.pi*80*t) + np.sin(2*np.pi*300*t) + 0.1*rng.standard_normal(t.size)

skip = x[::2]                  # 한 행 걸러 버리기 (필터 없음) → 500 Hz
dec = decimate(x, 2, ftype="fir", zero_phase=True)   # 저역 통과 후 버리기 → 500 Hz

for name, y in [("1000 Hz 원본", x), ("[::2] 건너뛰기", skip), ("decimate", dec)]:
    fs = fs_hi if y is x else 500
    f, p = welch(y, fs=fs, nperseg=fs)               # 1 Hz 간격
    top = f[np.argsort(p)[-2:]][::-1]
    print(f"{name:12s} fs={fs:4d}  가장 큰 두 봉우리: {top} Hz  | 200 Hz의 power: {p[f == 200][0]:.3f}")

### 예시 3 — 공통 사건으로 두 기기의 지연 추정하기

In [ ]:
import numpy as np
from scipy.signal import correlate, correlation_lags

fs = 500
t = np.arange(0, 60, 1/fs)
label = ((t >= 25) & (t < 35)).astype(int)
rng = np.random.default_rng(0)

# 공통 물리 사건: 발 디딤 충격. 걷는 동안 약 1초마다, 간격은 조금씩 불규칙(가정)
steps = np.cumsum(rng.uniform(0.85, 1.15, 80))
steps = steps[(steps < 60) & ~((steps >= 25) & (steps < 35))]     # FOG 중에는 디딤 없음(가정)
def impacts(times, width=0.03):
    x = np.zeros(t.size)
    for s in times:
        x += np.exp(-0.5 * ((t - s) / width) ** 2)
    return x

TRUE_LAG = 0.120                                   # 기기 B의 시계가 120 ms 늦다고 가정
a = impacts(steps) + 0.05 * rng.standard_normal(t.size)             # 기기 A (예: 정강이 IMU)
b = impacts(steps + TRUE_LAG) + 0.05 * rng.standard_normal(t.size)  # 기기 B (같은 충격을 늦게 기록)

def best_lag(a, b, fs, max_lag_s):
    """b가 a보다 몇 초 늦은가 (+면 b가 늦음). 탐색 범위를 ±max_lag_s로 제한한다."""
    c = correlate(b - b.mean(), a - a.mean(), mode="full")
    lags = correlation_lags(b.size, a.size, mode="full")
    keep = np.abs(lags) <= max_lag_s * fs
    c, lags = c[keep], lags[keep]
    i = np.argmax(c)
    r = c[i] / np.sqrt(((a - a.mean())**2).sum() * ((b - b.mean())**2).sum())
    return lags[i] / fs, r

print("참값: %.3f s" % TRUE_LAG)
print("추정(±0.5 s 탐색): %.3f s, r = %.2f" % best_lag(a, b, fs, 0.5))

# 걸음이 완벽히 규칙적이면? → 한 걸음 주기만큼 떨어진 곳에도 똑같이 높은 봉우리가 생긴다
reg = np.arange(0.5, 60, 1.0)
a2 = impacts(reg) + 0.05 * rng.standard_normal(t.size)
b2 = impacts(reg + TRUE_LAG) + 0.05 * rng.standard_normal(t.size)
for m in [0.5, 1.5]:
    print("규칙적 걸음, 탐색 ±%.1f s: 추정 %.3f s, r = %.2f" % ((m,) + best_lag(a2, b2, fs, m)))
lags = correlation_lags(t.size, t.size, mode="full")
for name, (x, y) in [("불규칙", (a, b)), ("규칙적", (a2, b2))]:
    c = correlate(y - y.mean(), x - x.mean(), mode="full")
    print("  %s 걸음: 지연 +0.12 / −0.88 / +1.12 s의 상관(최댓값 대비) = %s" % (
        name, " / ".join("%.2f" % (c[lags == round(L * fs)][0] / c.max()) for L in [0.12, -0.88, 1.12])))

### 예시 4 — 같은 탐지기, 다른 window 라벨 규칙, 다른 성능 숫자

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

# ---- 합성 데이터 뼈대 + 짧은 FOG 하나(48–50.5초, 가정) ----
fs = 500
t = np.arange(0, 60, 1/fs)
label = ((t >= 25) & (t < 35)).astype(int)
label[(t >= 48) & (t < 50.5)] = 1
rng = np.random.default_rng(0)
acc = np.where(label == 0, np.sin(2*np.pi*1.0*t), 0.4*np.sin(2*np.pi*5.0*t)) + 0.1*rng.standard_normal(t.size)

def window_starts(n, fs=500, win_s=2.0, step_s=1.0):
    """win_s초 window를 step_s초씩 옮길 때 각 window의 시작 행 (기본: 2초, 50% 겹침)"""
    w, s = int(win_s * fs), int(step_s * fs)
    return np.arange(0, n - w + 1, s)

def window_label(y_win, rule="majority"):
    if rule == "majority":                       # FOG 표본이 절반 이상이면 1
        return int(y_win.mean() >= 0.5)
    if rule == "any":                            # FOG 표본이 하나라도 있으면 1
        return int(y_win.max() == 1)
    if rule == "center":                         # 가운데 표본의 라벨
        return int(y_win[len(y_win) // 2])
    raise ValueError(rule)

def fi_score(x, fs):
    """한 window의 freeze index (Day I18의 정의를 점수로만 사용)"""
    x = x - x.mean()
    p = np.abs(np.fft.rfft(x)) ** 2
    f = np.fft.rfftfreq(x.size, 1 / fs)
    return p[(f >= 3) & (f <= 8)].sum() / p[(f >= 0.5) & (f < 3)].sum()

W = 2 * fs
starts = window_starts(t.size, fs)
score = np.array([fi_score(acc[s:s + W], fs) for s in starts])
frac = np.array([label[s:s + W].mean() for s in starts])
print("window 수:", len(starts), "| FOG가 일부만 걸친 window:", int(((frac > 0) & (frac < 1)).sum()))
rows = []
for rule in ["majority", "any", "center"]:
    y = np.array([window_label(label[s:s + W], rule) for s in starts])
    pred = score > 1.0
    rows.append({"rule": rule, "positives": int(y.sum()),
                 "sens": pred[y == 1].mean(), "spec": (~pred[y == 0]).mean(), "AUC": roc_auc_score(y, score)})
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("\n경계 window (시작초, FOG 비율, 점수):")
for s, fr, sc in zip(starts, frac, score):
    if 0 < fr < 1:
        print("  %4.1f  %.2f  %6.2f" % (s / fs, fr, sc))

## 직접 해보기

**1. 업샘플 지문 찾기.** 100 Hz 신호를 500 Hz로 보간한 뒤, 원본이 몇 행마다 그대로 들어 있는지 확인하세요.

In [ ]:
import numpy as np
from scipy.interpolate import CubicSpline
t100 = np.arange(0, 10, 1/100)
x100 = np.sin(2*np.pi*1.0*t100)
t500 = np.arange(0, t100[-1] + 1e-9, 1/500)
up = CubicSpline(t100, x100)(t500)
print(len(x100), len(up), np.allclose(up[::____], x100))

In [ ]:
print(len(x100), len(up), np.allclose(up[::5], x100))

**2. 올바른 내려 표본화.** 80 Hz와 400 Hz 성분이 있는 1000 Hz 신호를 500 Hz로 내리세요. 건너뛰기와 비교합니다.

In [ ]:
import numpy as np
from scipy.signal import welch, decimate
fs_hi = 1000
t = np.arange(0, 10, 1/fs_hi)
x = np.sin(2*np.pi*80*t) + np.sin(2*np.pi*400*t)
for name, y in [("[::2]", x[::2]), ("decimate", decimate(x, ____, ftype="fir", zero_phase=True))]:
    f, p = welch(y, fs=500, nperseg=500)
    print(name, "봉우리:", sorted(f[np.argsort(p)[-2:]].tolist()))

In [ ]:
("decimate", decimate(x, 2, ftype="fir", zero_phase=True))

**3. 지연의 부호 읽기.** 기기 B가 A보다 80 ms **이르게** 기록했습니다. 예시 3의 `best_lag`로 추정하고 부호를 해석하세요. 빈칸은 탐색 범위를 행 수로 바꾸는 부분입니다.

In [ ]:
import numpy as np
from scipy.signal import correlate, correlation_lags
fs = 500
t = np.arange(0, 60, 1/fs)
rng = np.random.default_rng(3)
steps = np.cumsum(rng.uniform(0.85, 1.15, 70)); steps = steps[steps < 59]
def impacts(times, width=0.03):
    return sum(np.exp(-0.5 * ((t - s) / width) ** 2) for s in times)
a = impacts(steps); b = impacts(steps - 0.08)          # B가 80 ms 이르다
def best_lag(a, b, fs, max_lag_s):
    c = correlate(b - b.mean(), a - a.mean(), mode="full")
    lags = correlation_lags(b.size, a.size, mode="full")
    keep = np.abs(lags) <= max_lag_s * ____
    return lags[keep][np.argmax(c[keep])] / fs
print("추정 지연: %.3f s" % best_lag(a, b, fs, 0.5))

In [ ]:
    keep = np.abs(lags) <= max_lag_s * fs

**4. window 개수 세기.** 60초(30000행) 기록을 2초 window, 1초 이동으로 자르면 window가 몇 개입니까? 3초 window, 0.3초 이동이면?

In [ ]:
import numpy as np
def window_starts(n, fs=500, win_s=2.0, step_s=1.0):
    w, s = int(win_s * fs), int(step_s * fs)
    return np.arange(0, n - w + 1, ____)
st = window_starts(30000)
print(len(st), st[:3], st[-1])
print(len(window_starts(30000, win_s=3.0, step_s=0.3)))

In [ ]:
    return np.arange(0, n - w + 1, s)

**5. "any" 규칙 구현.** 1000표본 window 중 마지막 300개만 FOG일 때 세 규칙의 라벨을 구하세요.

In [ ]:
import numpy as np
def window_label(y_win, rule="majority"):
    if rule == "majority": return int(y_win.mean() >= 0.5)
    if rule == "any":      return int(y_win.____() == 1)
    if rule == "center":   return int(y_win[len(y_win) // 2])
y_win = np.r_[np.zeros(700), np.ones(300)]           # 1000표본 중 마지막 300개만 FOG
print({r: window_label(y_win, r) for r in ["majority", "any", "center"]})

In [ ]:
    if rule == "any":      return int(y_win.max() == 1)

### 실제 데이터로 해보기 (선택)

**R1 — 원래 표본화율과 업샘플 지문.**

In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import welch

RAW = "data/xuanwu_raw/Raw/001"             # 원본 zip을 data/xuanwu_raw/에 푼 위치
FIL = "data/xuanwu/Filtered Data/001"

# 1) 원본 EEG/EMG 헤더: 채널 수와 표본 간격
hdr = open(f"{RAW}/1211-003.vhdr", encoding="utf-8", errors="replace").read().splitlines()
print([l for l in hdr if l.startswith(("NumberOfChannels", "SamplingInterval"))])

# 2) 원본 IMU 파일: 시각 간격으로 원래 표본화율 확인
imu = {}
for s in ["LShank", "Waist", "Arm"]:
    r = pd.read_csv(f"{RAW}/{s}.csv", header=None, usecols=range(8), skipinitialspace=True)
    ts = pd.to_datetime(r[0], format="%Y_%m_%d_%H_%M_%S_%f")
    dt = np.diff(ts.to_numpy()).astype("timedelta64[ms]").astype(int)
    print(f"{s:6s} 원본 {len(r)}행, 시각 간격 중앙값 {np.median(dt):.0f} ms")
    imu[s] = (r, ts)

# 3) 라벨된 파일의 IMU 7열이 원본 행과 같은가? (5행마다 = 보간의 매듭)
COL0 = {"LShank": 32, "Waist": 46, "Arm": 53}
f = pd.read_csv(f"{FIL}/task_1.txt", header=None, usecols=[1] + list(range(32, 60)))
print("라벨된 파일 시각:", f[1].iloc[0], "→", f[1].iloc[-1], f"({len(f)}행)")
for s, (r, ts) in imu.items():
    rv = r[list(range(1, 8))].to_numpy(float)
    k = f[list(range(COL0[s], COL0[s] + 7))].to_numpy()[::5]
    h = np.flatnonzero(np.isclose(rv, k[0], atol=1e-6).all(axis=1))[0]   # 첫 매듭과 같은 원본 행
    same = np.isclose(rv[h:h + len(k)], k, atol=1e-6).all(axis=1).mean()
    span = (ts.iloc[h + len(k) - 1] - ts.iloc[h]).total_seconds()
    print(f"{s:6s} 매듭 {len(k)}개 중 원본과 같은 비율 {same:.4f} | 시작 원본 행 {h} | 그 행의 원본 시각 {ts.iloc[h].time()} | 원본 시각으로 잰 길이 {span:.2f} s")

# 4) 스펙트럼: 원래 100 Hz였다면 50 Hz 위는 비어 있어야 한다
fs = 500
for name, c in [("LShank_ax", 32), ("Arm_SC", 59)]:
    x = f[c].to_numpy()
    fr, p = welch(x - x.mean(), fs=fs, nperseg=1000)
    tot = p[fr > 0.5].sum()
    print(f"{name:9s} power 비율: 16 Hz 위 {p[fr > 16].sum() / tot:.3f}, 50 Hz 위 {p[fr > 50].sum() / tot:.5f}")
e = pd.read_csv(f"{FIL}/task_1.txt", header=None, usecols=[28])[28].to_numpy()
fr, p = welch(e - e.mean(), fs=fs, nperseg=1000)
print(f"EMG(c28) power 비율: 50 Hz 위 {p[fr > 50].sum() / p[fr > 0.5].sum():.3f} | 49–51 Hz / 45–55 Hz 중앙값: {p[(fr >= 49) & (fr <= 51)].mean() / np.median(p[(fr > 45) & (fr < 55)]):.3f}")

**R2 — 허리와 정강이가 함께 본 발 디딤 충격으로 지연 추정.** 30초 구간마다 지연을 구해, 과제가 진행되며 지연이 0.5초 쪽으로 **흘러가는지** 봅니다.

In [ ]:
import numpy as np
import pandas as pd
from scipy.signal import butter, sosfiltfilt, correlate, correlation_lags

FIL = "data/xuanwu/Filtered Data/001"
fs = 500
d = pd.read_csv(f"{FIL}/task_1.txt", header=None, usecols=[28, 32, 33, 34, 46, 47, 48])

sos_hp = butter(2, 0.5, btype="high", fs=fs, output="sos")
sos_lp = butter(2, 5, btype="low", fs=fs, output="sos")
def impact_env(xyz):
    """3축 크기 → 0.5 Hz 고역(중력 제거) → 정류 → 5 Hz 저역: 발 디딤 충격의 덩어리"""
    m = np.linalg.norm(xyz, axis=1)
    return sosfiltfilt(sos_lp, np.abs(sosfiltfilt(sos_hp, m)))

def best_lag(a, b, fs, max_lag_s):
    """b가 a보다 몇 초 늦은가 (+면 b가 늦음), 탐색 범위 ±max_lag_s"""
    a, b = a - a.mean(), b - b.mean()
    c = correlate(b, a, mode="full")
    lags = correlation_lags(b.size, a.size, mode="full")
    keep = np.abs(lags) <= max_lag_s * fs
    i = np.argmax(c[keep])
    return lags[keep][i] / fs, c[keep][i] / np.sqrt((a**2).sum() * (b**2).sum())

shank = impact_env(d[[32, 33, 34]].to_numpy())
waist = impact_env(d[[46, 47, 48]].to_numpy())
emg = sosfiltfilt(sos_lp, np.abs(d[28].to_numpy() - d[28].mean()))    # L-TA EMG 크기 덩어리
W = 30 * fs
print(" 구간(s)  허리-정강이 지연(ms)  r   | EMG-정강이 지연(ms)  r")
for s in range(0, len(shank) - W + 1, W):
    sl = slice(s, s + W)
    l1, r1 = best_lag(shank[sl], waist[sl], fs, 0.25)
    l2, r2 = best_lag(shank[sl], emg[sl], fs, 0.25)
    print(f"{s // fs:4d}–{(s + W) // fs:<4d}     {1000 * l1:+6.0f}          {r1:.2f} |   {1000 * l2:+6.0f}            {r2:.2f}")

**R3 — window 라벨 규칙이 실제 숫자를 얼마나 바꾸나.** 모든 기록에서 있는 쪽 정강이의 가속도 크기로 window별 freeze index를 구하고, 규칙마다 AUC를 계산합니다. 2초/1초(이 파트의 기본)와 3초/0.3초(논문 설정)를 비교합니다.

In [ ]:
import glob, os
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

DATA = "data/xuanwu/Filtered Data"
fs = 500
SHANK = {"L": [32, 33, 34], "R": [39, 40, 41]}

def window_starts(n, fs=500, win_s=2.0, step_s=1.0):
    w, s = int(win_s * fs), int(step_s * fs)
    return np.arange(0, n - w + 1, s)

recs = []                                          # (기록 이름, 있는 쪽 정강이 크기, 라벨)
for path in sorted(glob.glob(f"{DATA}/**/task_*.txt", recursive=True)):
    rec = os.path.dirname(os.path.relpath(path, DATA))          # 001 … 또는 008/OFF_1
    d = pd.read_csv(path, header=None, usecols=[60] + SHANK["L"] + SHANK["R"])
    side = "L" if (d[SHANK["L"]] != 0).any(axis=None) else "R"   # 있는 쪽 정강이
    recs.append((rec, np.linalg.norm(d[SHANK[side]].to_numpy(), axis=1), d[60].to_numpy()))

for win_s, step_s in [(2.0, 1.0), (3.0, 0.3)]:     # 이 Day의 기본값, 그리고 논문이 쓴 설정
    W = int(win_s * fs)
    f = np.fft.rfftfreq(W, 1 / fs)
    LOCO, FRZ = (f >= 0.5) & (f < 3), (f >= 3) & (f <= 8)
    parts = []
    for rec, mag, y in recs:
        st = window_starts(len(y), fs, win_s, step_s)
        seg = mag[st[:, None] + np.arange(W)]
        seg = seg - seg.mean(axis=1, keepdims=True)
        p = np.abs(np.fft.rfft(seg, axis=1)) ** 2
        lab = y[st[:, None] + np.arange(W)]
        parts.append(pd.DataFrame({"patient": rec[:3], "fi": p[:, FRZ].sum(1) / p[:, LOCO].sum(1),
                                   "frac": lab.mean(1), "center": lab[:, W // 2]}))
    win = pd.concat(parts, ignore_index=True)
    rules = {"majority (≥50%)": win["frac"] >= 0.5, "any (>0%)": win["frac"] > 0,
             "center": win["center"] == 1, "PFG ≥ 80%": win["frac"] >= 0.8}
    print(f"\n== {win_s:.0f}초 window, {step_s}초 이동: window {len(win)}개, "
          f"FOG가 일부만 걸친 window {100 * ((win.frac > 0) & (win.frac < 1)).mean():.1f}%")
    rows = []
    for name, y in rules.items():
        y = y.to_numpy()
        per = [roc_auc_score(y[g.index], g["fi"]) for _, g in win.groupby("patient") if 0 < y[g.index].mean() < 1]
        rows.append({"rule": name, "FOG window %": 100 * y.mean(), "AUC pooled": roc_auc_score(y, win["fi"]),
                     "AUC 환자별 중앙값": np.median(per), "환자 수": len(per)})
    print(pd.DataFrame(rows).round(3).to_string(index=False))